# Photodetectors: ideal, diode, and carrier dynamics

Three notebook-local Circulax components, all sharing `optical_in`, `anode`, `cathode` ports:

| Model | Optical conversion | Electrical I-V/C-V | Internal state |
|---|---|---|---|
| `IdealPhotodetector` | instantaneous responsivity | ideal current source | none |
| `Photodiode` | instantaneous responsivity | dark I-V + junction C-V | junction charge |
| `CarrierPhotodiode` | carrier generation/collection | same I-V/C-V as above | electron/hole populations, screening, recombination |

Parameters here are illustrative — not fit to a real device.

Optical terminals use the standard envelope reference impedance (1 Ω) and have a wavelength-dependent complex reflection coefficient, so terminal field, incident field, and absorbed power are all different quantities. Each source in the testbench below launches a fixed incident power at its own detector and absorbs whatever reflects back. Current convention: positive into the anode, so illumination gives negative anode current. All three models are non-holomorphic (intensity-dependent), which matters for the AC solver later on.

Measurements below use `dc()` for operating points and I–V, `sp()` for C–V and small-signal gain, `transient()` for pulses, and `hb()` for periodic modulation. Sources, loads and grounds are attached with `attach_testbench()`. The compact-model equations define the physics and supply reference checks.


In [ ]:
import diffrax
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from circulax import attach_testbench, compile_circuit
from circulax.components.base_component import PhysicsReturn, Signals, component, source
from circulax.components.electronic import Resistor, VoltageSource

jax.config.update("jax_enable_x64", True)
plt.rcParams.update({"figure.figsize": (10, 3.8), "axes.grid": True})
QE = 1.602176634e-19  # C
PORTS = ("optical_in", "anode", "cathode")
RESPONSIVITY = 0.8  # A/W
REFERENCE_WAVELENGTH_NM = 1550.0
REFLECTIVITY = 0.04  # reflected fraction of incident power at the reference wavelength
EXTERNAL_RESPONSIVITY = RESPONSIVITY * (1 - REFLECTIVITY)


## Wavelength-dependent responsivity and reflection

All three models take a `wavelength_nm` param — the optical carrier wavelength of a narrowband envelope. The transient time axis is modulation time, not the carrier oscillation.

Absorbed-power responsivity profile used here:

$$
\mathcal R_\mathrm{abs}(\lambda)=\mathcal R_0\frac{\lambda}{\lambda_0}
\exp[-\tfrac12((\lambda-\lambda_0)/\sigma_R)^2].
$$

The $\lambda/\lambda_0$ term is photon energy at fixed quantum efficiency; the Gaussian is a stand-in for real spectral efficiency. `responsivity` is this curve's value at `center_wavelength_nm`, in A/W of **absorbed** power — before any collection loss. Swap in a measured curve whenever you have one. Sanity bound: at unity quantum efficiency, absorbed-power responsivity is $q_e\lambda/(hc)$; the default profile stays under it.

Reflection is a complex amplitude, not a current tacked onto the port:

$$
r(\lambda)=\sqrt{\rho_0}\exp[-\tfrac14((\lambda-\lambda_0)/\sigma_r)^2]
\exp[j(\phi_0+k_\phi(\lambda-\lambda_0))],\qquad
Y_\mathrm{opt}(\lambda)=\frac{1-r}{1+r}\quad (Z_0=1).
$$

`reflectivity` ($\rho_0$) is power reflectance at the reference wavelength; `reflection_phase_rad`/`reflection_phase_slope_rad_nm` set the field phase. Given incident field $a$ and reflected $b=ra$, terminal field $E=a+b$:

$$
a=E/(1+r),\quad P_\mathrm{inc}=|a|^2,\quad
P_\mathrm{refl}=|r|^2P_\mathrm{inc},\quad
P_\mathrm{abs}=(1-|r|^2)P_\mathrm{inc},\quad
I_\mathrm{ph}=\mathcal R_\mathrm{abs}(\lambda)P_\mathrm{abs}.
$$

This is a one-port model: it absorbs everything that isn't reflected, no transmission port. Keep wavelengths/bandwidths positive, `responsivity >= 0`, `0 <= reflectivity < 1`. Defaults: 4% reflectance, 0.768 A/W external responsivity at 1550 nm (0.8 × 0.96). If you're starting from a measured **external** responsivity, divide by $1-|r|^2$ first to back out the absorbed-power curve — otherwise you double-count the reflection loss.

## 1. Ideal detector

Straight optical-to-electrical conversion using the shared spectral response and optical admittance. Equal and opposite stamps on anode/cathode keep terminal current conserved. `reflectivity=0` gives you the matched-detector limit.


In [ ]:
def spectral_response(responsivity=0.8, wavelength_nm=1550.0, center_wavelength_nm=1550.0,
                      responsivity_bandwidth_nm=250.0, reflectivity=0.04,
                      reflection_bandwidth_nm=180.0, reflection_phase_rad=0.0,
                      reflection_phase_slope_rad_nm=0.01):
    detuning_nm = wavelength_nm - center_wavelength_nm
    r_abs = responsivity * (wavelength_nm / center_wavelength_nm)
    r_abs *= jnp.exp(-0.5 * (detuning_nm / responsivity_bandwidth_nm)**2)
    reflection = jnp.sqrt(reflectivity) * jnp.exp(-0.25 * (detuning_nm / reflection_bandwidth_nm)**2)
    reflection = reflection * jnp.exp(1j * (reflection_phase_rad + reflection_phase_slope_rad_nm * detuning_nm))
    return r_abs, reflection


def optical_conversion(field, responsivity=0.8, wavelength_nm=1550.0, center_wavelength_nm=1550.0,
                       responsivity_bandwidth_nm=250.0, reflectivity=0.04,
                       reflection_bandwidth_nm=180.0, reflection_phase_rad=0.0,
                       reflection_phase_slope_rad_nm=0.01):
    r_abs, reflection = spectral_response(
        responsivity, wavelength_nm, center_wavelength_nm, responsivity_bandwidth_nm,
        reflectivity, reflection_bandwidth_nm, reflection_phase_rad, reflection_phase_slope_rad_nm)
    optical_current = (1 - reflection) / (1 + reflection) * field
    incident = field / (1 + reflection)
    absorbed_power = (1 - jnp.abs(reflection)**2) * jnp.abs(incident)**2
    return optical_current, r_abs * absorbed_power


@component(ports=PORTS, holomorphic=False)
def IdealPhotodetector(
    signals: Signals, responsivity: float = 0.8,
    wavelength_nm: float = 1550.0, center_wavelength_nm: float = 1550.0,
    responsivity_bandwidth_nm: float = 250.0, reflectivity: float = 0.04,
    reflection_bandwidth_nm: float = 180.0, reflection_phase_rad: float = 0.0,
    reflection_phase_slope_rad_nm: float = 0.01,
) -> PhysicsReturn:
    optical_current, i_photo = optical_conversion(signals.optical_in,
        responsivity, wavelength_nm, center_wavelength_nm, responsivity_bandwidth_nm,
        reflectivity, reflection_bandwidth_nm, reflection_phase_rad, reflection_phase_slope_rad_nm)
    return {"optical_in": optical_current,
            "anode": -i_photo, "cathode": i_photo}, {}



## 2. Diode I-V, C-V, and dark current

With $V=V_a-V_c$:

$$
I_\mathrm{dark}(V)=I_s\operatorname{expm1}(V/(n_dV_T))
 +I_\mathrm{leak}\tanh(V/V_\mathrm{leak})+V/R_\mathrm{sh}.
$$

`i_leak` is *additional* reverse leakage on top of `i_s` and the shunt term — don't double-count total dark current by also bumping `i_s`.  This profile doesn't model avalanche breakdown, so stay inside the plotted range.

Depletion capacitance $C(V)=C_0(1-V/V_\mathrm{bi})^{-1/2}$, integrated to charge:

$$
Q(V)=2C_0V_\mathrm{bi}\left(1-\sqrt{1-V/V_\mathrm{bi}}\right).
$$

Past `fc * v_bi` we switch to a quadratic continuation so Q, C, and dC/dV all stay continuous and we don't hit the depletion singularity. Circulax wants **Q**, not `C(V)*V` — the latter gives the wrong differential capacitance. If you plug in a measured C-V curve, integrate it the same way before handing it to the solver.

The I–V curve below comes from a voltage-clamp DC testbench; source branch current measures detector current. C–V comes from the imaginary admittance recovered from `sp()` in a separate biased, loaded testbench. The analytic equations are retained only as reference assertions.


In [ ]:
def dark_iv(v, i_s=1e-12, ideality=1.5, v_t=0.02585,
            i_leak=1e-9, v_leak=0.2, r_sh=1e12):
    return i_s * jnp.expm1(v / (ideality * v_t)) + i_leak * jnp.tanh(v / v_leak) + v / r_sh


def junction_charge(v, c0=20e-15, v_bi=0.8, fc=0.5):
    v_join = fc * v_bi
    # A complex V is retained for the electrical imaginary-block equations.
    v_depletion = jnp.where(jnp.real(v) <= v_join, v, v_join)
    q_depletion = 2 * c0 * v_bi * (1 - jnp.sqrt(1 - v_depletion / v_bi))
    dv = jnp.where(jnp.real(v) > v_join, v - v_join, 0.0)
    c_join = c0 / jnp.sqrt(1 - fc)
    dc_join = c0 / (2 * v_bi * (1 - fc)**1.5)
    return q_depletion + c_join * dv + 0.5 * dc_join * dv**2


@component(ports=PORTS, holomorphic=False)
def Photodiode(signals: Signals, responsivity: float = 0.8,
    wavelength_nm: float = 1550.0, center_wavelength_nm: float = 1550.0,
    responsivity_bandwidth_nm: float = 250.0, reflectivity: float = 0.04,
    reflection_bandwidth_nm: float = 180.0, reflection_phase_rad: float = 0.0,
    reflection_phase_slope_rad_nm: float = 0.01,
               i_s: float = 1e-12, ideality: float = 1.5, v_t: float = 0.02585,
               i_leak: float = 1e-9, v_leak: float = 0.2, r_sh: float = 1e12,
               c0: float = 20e-15, v_bi: float = 0.8, fc: float = 0.5) -> PhysicsReturn:
    v = signals.anode - signals.cathode
    current = dark_iv(v, i_s, ideality, v_t, i_leak, v_leak, r_sh)
    optical_current, i_photo = optical_conversion(signals.optical_in,
        responsivity, wavelength_nm, center_wavelength_nm, responsivity_bandwidth_nm,
        reflectivity, reflection_bandwidth_nm, reflection_phase_rad, reflection_phase_slope_rad_nm)
    current -= i_photo
    charge = junction_charge(v, c0, v_bi, fc)
    return {"optical_in": optical_current, "anode": current, "cathode": -current}, {
        "anode": charge, "cathode": -charge}


@source(ports=("p1", "p2"), holomorphic=True)
def OpticalCW(signals: Signals, t: float, power: float = 1e-6,  # noqa: N802
              phase: float = 0.0, modulation: float = 0.0, frequency: float = 1e9) -> PhysicsReturn:
    # Sinusoidal incident POWER, not sinusoidal field amplitude.
    incident_power = power * (1 + modulation * jnp.cos(2 * jnp.pi * frequency * t))
    incident = jnp.sqrt(incident_power) * jnp.exp(1j * phase)
    current = signals.p1 - signals.p2 - 2 * incident
    return {"p1": current, "p2": -current}, {}


probe_device = {
    "instances": {"PD": {"component": "diode"}},
    "ports": {port: f"PD,{port}" for port in PORTS},
}
probe_models = {"diode": Photodiode, "light": OpticalCW,
                "bias": VoltageSource, "load": Resistor, "ground": lambda: 0}
iv_netlist = attach_testbench(
    probe_device,
    sources={"optical_in": {"name": "LIGHT", "component": "light"},
             "anode": {"name": "CLAMP", "component": "bias"}},
    gnd=["cathode"],
)
iv_circuit = compile_circuit(iv_netlist, probe_models, is_complex=True, backend="dense")
v_iv = jnp.linspace(-5.0, 0.65, 400)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
iv_powers = jnp.array([0.0, 1e-6, 1e-3])
# dc() already vmaps array-valued voltage parameters; the outer vmap adds
# an independent power axis, giving a Cartesian (power, voltage) sweep.
iv_ops = jax.vmap(lambda power: iv_circuit.dc(
    params={"CLAMP.V": v_iv, "power": power}, rtol=1e-10, atol=1e-13,
))(iv_powers)
measured_iv = -jnp.real(iv_circuit.port(iv_ops, "CLAMP,i_src"))
np.testing.assert_allclose(measured_iv,
    dark_iv(v_iv)[None, :] - EXTERNAL_RESPONSIVITY * iv_powers[:, None],
    rtol=1e-6, atol=1e-12)
# Python loops below only format and plot the already solved arrays.
for power, measured_current in zip(np.asarray(iv_powers), measured_iv, strict=True):
    axes[0].plot(v_iv, measured_current, label=f"P = {power:g} W")
axes[0].set(xlabel="Anode − cathode voltage (V)", ylabel="Anode current (A)")
axes[0].set_yscale("symlog", linthresh=1e-10)
axes[0].legend()

# A finite load leaves the anode available for an SP measurement. The DC
# cathode supply is an AC ground; the actual junction voltage includes load drop.
cv_netlist = attach_testbench(
    probe_device,
    sources={"optical_in": {"name": "LIGHT", "component": "light", "settings": {"power": 0.0}},
             "cathode": {"name": "BIAS", "component": "bias"}},
    loads={"anode": {"component": "load", "settings": {"R": 50.0}}},
)
cv_circuit = compile_circuit(cv_netlist, probe_models, is_complex=True, backend="dense")
cv_frequency = 1e9


def measure_capacitance(supply):
    params = {"BIAS.V": supply}
    op = cv_circuit.dc(params=params, rtol=1e-10, atol=1e-13)
    s11 = cv_circuit.sp(ports="PD,anode", freqs=jnp.array([cv_frequency]),
                        z0=50.0, y_dc=op, params=params)[0, 0, 0]
    admittance = (1 - s11) / (50 * (1 + s11))
    voltage = jnp.real(cv_circuit.port(op, "PD,anode") - cv_circuit.port(op, "PD,cathode"))
    return voltage, jnp.imag(admittance) / (2 * jnp.pi * cv_frequency)


v_cv, capacitance = jax.vmap(measure_capacitance)(-jnp.linspace(-5.0, 0.6, 400))
axes[1].plot(v_cv, capacitance * 1e15)
axes[1].set(xlabel="Solved anode - cathode voltage (V)", ylabel="SP differential capacitance (fF)")
plt.tight_layout()
plt.show()
# Model equations remain independent reference checks, not the plotted measurement.
np.testing.assert_allclose(capacitance, jax.vmap(jax.grad(junction_charge))(v_cv), rtol=1e-7)
assert bool(jnp.all(capacitance > 0))


## 3. Electron and hole populations

Excess concentrations $n=N_0x_e$, $p=N_0x_h$ — normalized states `x_e`, `x_h` so the circuit unknowns stay near 1. For active volume $\mathcal V$, responsivity gives an effective pair-generation rate:

$$
g(P)=\frac{\mathcal R_\mathrm{abs}(\lambda)P_\mathrm{abs}}{q_e\mathcal V N_0},\qquad
\dot x_e=g-x_e/\tau_e-r,\qquad
\dot x_h=g-x_h/\tau_h-r,\qquad
r=x_ex_h/\tau_\mathrm{rec}.
$$

Absorbed-power responsivity here is pre-collection-loss; reflection is already applied before generation. Recombination pulls pairs from both populations equally. To fake a reduced collection field:

$$
\tau_{e,h}=\tau_{e,h,0}\frac{V_\mathrm{bi}+V_\mathrm{ref}}
 {V_\mathrm{bi}+\max(-V,0)}
 \left[1+s\frac{\max(x_e,0)+\max(x_h,0)}{2x_\mathrm{screen}}\right].
$$

Same thing written out with an explicit depletion width $L=1\,\mu\mathrm m$:

$$
E_0=(V_\mathrm{bi}+\max(-V,0))/L,\quad
E_\mathrm{eff}=E_0/[1+s(x_e+x_h)/(2x_\mathrm{screen})],\quad
\tau_{e,h}=\tau_{e,h,0}E_\mathrm{ref}/E_\mathrm{eff}.
$$

**The feedback loop:** more carriers → weaker internal field → slower sweep-out → more carriers hang around. Generation feeds carriers in, recombination caps how high they pile up. The field is just an algebraic function of carrier state and junction voltage — not a separate dynamic variable.

This screening law is a phenomenological hack: a spatially-averaged pair of populations can't actually reconstruct a real space-charge field. If you want to justify the functional form, look at [self-consistent transport simulations](https://www.sciencedirect.com/science/article/abs/pii/S0038110100000952), but fit the parameters to real impulse response data before trusting predictions from it. The reverse-bias branch and population clamps are only piecewise differentiable — all the examples below stay in reverse bias to avoid that edge.

Terminal photocurrent is a weighted transport sum:

$$
I_\mathrm{ph}=q_e\mathcal V N_0
 [w_e x_e/\tau_e+(1-w_e)x_h/\tau_h].
$$

`electron_weight = 0.5` here; weights sum to 1 so full collection gives one electron charge per pair (not two). Think of them as weighting-field travel fractions, not independent terminal currents. Fixed sweep-out times alone would just be linear filtering — making sweep-out density-dependent is what produces peak compression and a longer tail; recombination on top of that also eats collected charge.

Circulax solves `f + dQ/dt = 0`, so the state rows stamp `tau_ref * (loss - generation)` into f and `tau_ref * x` into Q. These are scaled rate-equation variables, not extra junction charge.


In [ ]:
def internal_field(x_e, x_h, v, v_bi=0.8, screening=1.0,
                   x_screen=1.0, depletion_width_m=1e-6):
    population = (jnp.maximum(jnp.real(x_e), 0) + jnp.maximum(jnp.real(x_h), 0)) / 2
    reverse_bias = jnp.maximum(-jnp.real(v), 0)
    unscreened = (v_bi + reverse_bias) / depletion_width_m
    effective = unscreened / (1 + screening * population / x_screen)
    return unscreened, effective


def carrier_rates(x_e, x_h, v, tau_e0=8e-12, tau_h0=40e-12,
                  v_bi=0.8, v_ref=3.0, screening=1.0, x_screen=1.0,
                  tau_rec=500e-12):
    _, effective_field = internal_field(x_e, x_h, v, v_bi, screening, x_screen)
    reference_field = (v_bi + v_ref) / 1e-6
    tau_e, tau_h = tau_e0 * reference_field / effective_field, tau_h0 * reference_field / effective_field
    recombination = x_e * x_h / tau_rec
    return tau_e, tau_h, recombination


@component(ports=PORTS, states=("x_e", "x_h"), holomorphic=False)
def CarrierPhotodiode(
    signals: Signals, responsivity: float = 0.8,
    wavelength_nm: float = 1550.0, center_wavelength_nm: float = 1550.0,
    responsivity_bandwidth_nm: float = 250.0, reflectivity: float = 0.04,
    reflection_bandwidth_nm: float = 180.0, reflection_phase_rad: float = 0.0,
    reflection_phase_slope_rad_nm: float = 0.01,
    i_s: float = 1e-12, ideality: float = 1.5, v_t: float = 0.02585,
    i_leak: float = 1e-9, v_leak: float = 0.2, r_sh: float = 1e12,
    c0: float = 20e-15, v_bi: float = 0.8, fc: float = 0.5,
    volume_m3: float = 100e-18, density_scale_m3: float = 1e21,
    tau_e0: float = 8e-12, tau_h0: float = 40e-12, v_ref: float = 3.0,
    screening: float = 1.0, x_screen: float = 1.0, tau_rec: float = 500e-12,
    electron_weight: float = 0.5, tau_ref: float = 20e-12,
) -> PhysicsReturn:
    v = signals.anode - signals.cathode
    x_e, x_h = signals.x_e, signals.x_h
    charge_scale = QE * volume_m3 * density_scale_m3
    optical_current, generation_current = optical_conversion(signals.optical_in,
        responsivity, wavelength_nm, center_wavelength_nm, responsivity_bandwidth_nm,
        reflectivity, reflection_bandwidth_nm, reflection_phase_rad, reflection_phase_slope_rad_nm)
    generation = generation_current / charge_scale
    tau_e, tau_h, recombination = carrier_rates(
        x_e, x_h, v, tau_e0, tau_h0, v_bi, v_ref, screening, x_screen, tau_rec)
    i_photo = charge_scale * (electron_weight * x_e / tau_e + (1 - electron_weight) * x_h / tau_h)
    current = dark_iv(v, i_s, ideality, v_t, i_leak, v_leak, r_sh) - i_photo
    charge = junction_charge(v, c0, v_bi, fc)
    return {
        "optical_in": optical_current, "anode": current, "cathode": -current,
        "x_e": tau_ref * (x_e / tau_e + recombination - generation),
        "x_h": tau_ref * (x_h / tau_h + recombination - generation),
    }, {"anode": charge, "cathode": -charge,
        "x_e": tau_ref * x_e, "x_h": tau_ref * x_h}

## A common circuit testbench

Three independent matched optical sources, each launching the **same incident power** at its own detector — think of it as three parallel experiments, not a power splitter. Every source absorbs its own reflection. A source stamps `I = E - 2a`, each detector stamps `I = Y_opt E`; solving the shared node gives `E = (1 + r)a`, which keeps the incident power fixed even as reflectivity changes with wavelength. Cathodes are reverse-biased, anodes drive 50 Ω to ground. The ideal detector has no dark current; the two diode models share the same electrical profile.

Stimulus: a Gaussian optical power pulse, 12 ps σ (28.3 ps FWHM), as an impulse approximation. We run out to 1.5 ns so even the screened carrier tail fully integrates. Outputs are in volts; divide by 50 Ω for load current.

`make_testbench()` uses `attach_testbench()` to terminate the device ports, keeping the detector netlist reusable for pulse, CW, and periodic stimuli.


In [ ]:
@source(ports=("p1", "p2"), holomorphic=True)
def OpticalGaussianPulse(signals: Signals, t: float, power: float = 1e-3,
                         center: float = 120e-12, sigma: float = 12e-12,
                         phase: float = 0.0) -> PhysicsReturn:
    incident = jnp.sqrt(power) * jnp.exp(-0.25 * ((t - center) / sigma)**2 + 1j * phase)
    current = signals.p1 - signals.p2 - 2 * incident
    return {"p1": current, "p2": -current}, {}


models = {"ground": lambda: 0, "optical_pulse": OpticalGaussianPulse,
          "bias": VoltageSource, "load": Resistor,
          "ideal": IdealPhotodetector, "diode": Photodiode, "carrier": CarrierPhotodiode}
detector_names = ("ideal", "diode", "carrier")
device_netlist = {
    "instances": {f"PD_{name}": {"component": name} for name in detector_names},
    "connections": {"PD_ideal,cathode": ("PD_diode,cathode", "PD_carrier,cathode")},
    "ports": {"bias": "PD_ideal,cathode",
              **{f"light_{name}": f"PD_{name},optical_in" for name in detector_names},
              **{name: f"PD_{name},anode" for name in detector_names}},
}


def make_testbench(device=device_netlist, optical_source="optical_pulse"):
    bench = attach_testbench(
        device,
        sources={"bias": {"name": "BIAS", "component": "bias", "settings": {"V": 3.0}},
                 **{f"light_{name}": {"name": f"LIGHT_{name}", "component": optical_source}
                    for name in detector_names}},
        loads={name: {"name": f"R_{name}", "component": "load", "settings": {"R": 50.0}}
               for name in detector_names},
        gnd=[port for port in device["ports"] if port.startswith("pad_ground_")],
    )
    # Re-expose measurement aliases after attach_testbench terminates the ports.
    bench["ports"] = {"light": "LIGHT_ideal,p1",
                      **{name: f"PD_{name},anode" for name in detector_names}}
    return bench


netlist = make_testbench()
circuit = compile_circuit(netlist, models, is_complex=True, backend="dense", rtol=1e-8, atol=1e-10)
times = jnp.linspace(0, 1.5e-9, 1501)


def simulate_waveforms(power=1e-3, bias=3.0, screening=1.0, tau_rec=500e-12,
             tau_e0=8e-12, tau_h0=40e-12, dt=1e-12, wavelength_nm=1550.0, reflectivity=0.04, phase=0.0):
    updates = {"power": power, "BIAS.V": bias, "wavelength_nm": wavelength_nm,
               "reflectivity": reflectivity, "phase": phase,
               "PD_carrier.screening": screening, "PD_carrier.tau_rec": tau_rec,
               "PD_carrier.tau_e0": tau_e0, "PD_carrier.tau_h0": tau_h0}
    y0 = circuit.dc(params=updates)
    solution = circuit.transient(t0=0, t1=float(times[-1]), dt0=dt, y0=y0,
                                 params=updates, saveat=diffrax.SaveAt(ts=times),
                                 max_steps=5000, throw=True)
    currents = {name: jnp.real((circuit.port(solution.ys, name) - circuit.port(solution.ys, "GND,p1"))
                                - (circuit.port(y0, name) - circuit.port(y0, "GND,p1"))) / 50
                for name in ("ideal", "diode", "carrier")}
    concentrations = {state: jnp.real(circuit.port(solution.ys, f"PD_carrier,{state}")) * 1e21
                      for state in ("x_e", "x_h")}
    junction_v = circuit.port(solution.ys, "carrier") - circuit.port(solution.ys, "BIAS,p1")
    field0, field = internal_field(concentrations["x_e"] / 1e21, concentrations["x_h"] / 1e21,
                                   junction_v, screening=screening)
    concentrations["field_unscreened"] = field0
    concentrations["field_screened"] = field
    concentrations["electrical_imaginary_max"] = jnp.max(jnp.stack([
        jnp.max(jnp.abs(jnp.imag(circuit.port(solution.ys, name)))) for name in detector_names
    ]))
    return currents, concentrations


def check_waveforms(result):
    current_set, states = result
    assert all(np.all(np.isfinite(value)) for value in (*current_set.values(), *states.values()))
    assert np.max(states["electrical_imaginary_max"]) < 1e-9


def simulate(**settings):
    result = simulate_waveforms(**settings)
    check_waveforms(result)
    return result


currents, concentrations = simulate()
t_ps = np.asarray(times) * 1e12
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for name, current in currents.items():
    axes[0].plot(t_ps, current * 1e3, label=name)
axes[0].set(xlabel="Time (ps)", ylabel="Baseline-subtracted load current (mA)", xlim=(60, 600))
axes[0].legend()
for state in ("x_e", "x_h"):
    density = concentrations[state]
    axes[1].plot(t_ps, density / 1e6, label="electrons" if state == "x_e" else "holes")
axes[1].set(xlabel="Time (ps)", ylabel="Excess concentration (cm⁻³)", xlim=(60, 600))
axes[1].legend()
plt.tight_layout()
plt.show()

## Saturation, bias dependence, and asymmetric response

Compare normalized responses across pulse powers to separate linear filtering from power-dependent dynamics. 10-90% rise and 90-10% fall times are reported from interpolated crossings around the peak — these are shape metrics for this particular Gaussian stimulus, not intrinsic device numbers.

The power and bias sweeps use `jax.vmap(simulate_waveforms)` to batch complete DC/transient experiments. The waveform function keeps its outputs in JAX; NumPy checks and shape metrics run after the batched solve.


In [ ]:
def crossing_time(t, y, level):
    hits = np.flatnonzero((y[:-1] - level) * (y[1:] - level) <= 0)
    if not len(hits):
        return np.nan
    k = hits[0]
    if y[k + 1] == y[k]:
        return t[k]
    return t[k] + (level - y[k]) * (t[k + 1] - t[k]) / (y[k + 1] - y[k])


def pulse_metrics(current, power, wavelength_nm=1550.0, reflectivity=0.04):
    t = np.asarray(times)
    k = int(np.argmax(current))
    peak = current[k]
    rise = crossing_time(t[:k + 1], current[:k + 1], 0.9 * peak) - crossing_time(t[:k + 1], current[:k + 1], 0.1 * peak)
    fall = crossing_time(t[k:], current[k:], 0.1 * peak) - crossing_time(t[k:], current[k:], 0.9 * peak)
    r_abs, reflection = spectral_response(wavelength_nm=wavelength_nm, reflectivity=reflectivity)
    external_responsivity = float(r_abs * (1 - jnp.abs(reflection)**2))
    incident_charge = external_responsivity * power * np.sqrt(2 * np.pi) * 12e-12
    return peak, np.trapezoid(current, t) / incident_charge, rise * 1e12, fall * 1e12


powers = np.array([1e-5, 1e-3, 1e-2])
pulse_sweep = jax.vmap(lambda power: simulate_waveforms(power=power))(jnp.asarray(powers))
check_waveforms(pulse_sweep)
sweep = {power: jax.tree.map(lambda values, k=k: values[k], pulse_sweep)
         for k, power in enumerate(powers)}
np.testing.assert_allclose(sweep[1e-3][0]["carrier"], currents["carrier"], rtol=1e-6, atol=1e-10)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
print("Peak P (mW)   Peak I (mA)   Charge fraction   Rise (ps)   Fall (ps)")
for power in powers:
    current = sweep[power][0]["carrier"]
    peak, fraction, rise, fall = pulse_metrics(current, power)
    print(f"{power * 1e3:11.3g} {peak * 1e3:13.4g} {fraction:17.4f} {rise:11.2f} {fall:11.2f}")
    axes[0].plot(t_ps, current / (EXTERNAL_RESPONSIVITY * power), label=f"{power * 1e3:g} mW")
    axes[1].plot(t_ps, current / peak, label=f"{power * 1e3:g} mW")
axes[0].set(xlabel="Time (ps)", ylabel="I / (external responsivity × peak power)", xlim=(60, 800))
axes[1].set(xlabel="Time (ps)", ylabel="I / peak I", xlim=(60, 800))
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

pulse_biases = jnp.array([1.0, 3.0])
bias_pulse_sweep = jax.vmap(lambda bias: simulate_waveforms(power=1e-3, bias=bias))(pulse_biases)
check_waveforms(bias_pulse_sweep)
low_bias = jax.tree.map(lambda values: values[0], bias_pulse_sweep[0])
np.testing.assert_allclose(bias_pulse_sweep[0]["carrier"][1], currents["carrier"], rtol=1e-6, atol=1e-10)
plt.figure()
plt.plot(t_ps, currents["carrier"] * 1e3, label="3 V reverse bias")
plt.plot(t_ps, low_bias["carrier"] * 1e3, label="1 V reverse bias")
plt.xlim(60, 800)
plt.xlabel("Time (ps)")
plt.ylabel("Load current (mA)")
plt.legend()
plt.tight_layout()
plt.show()

## Isolating the nonlinear feedback

Recombination is set negligible in both runs here so we isolate screening cleanly. At high power, compare screening on vs. off with everything else — bias, sweep-out times, diode profile, load — held fixed. With screening on, the field collapses as carriers build up, sweep-out slows, and you get a long tail. Note peak compression can happen even while almost all the charge eventually gets collected — that's a different effect from recombination loss.

The optical stimulus itself is symmetric; separate electron/hole sweep-out times already give a faster rise than fall even at low power. What we're isolating below is the *extra* asymmetry that screening adds on top of that, with both threshold intervals marked explicitly in the last plot.


In [ ]:
feedback_off, off_states = simulate(power=1e-2, screening=0.0, tau_rec=1e6)
feedback_on, on_states = simulate(power=1e-2, screening=1.0, tau_rec=1e6)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for label, current_set, states in [("screening off", feedback_off, off_states),
                                   ("screening on", feedback_on, on_states)]:
    axes[0].plot(t_ps, current_set["carrier"] * 1e3, label=label)
    axes[1].plot(t_ps, states["field_screened"] / 1e6, label=label)
    axes[2].plot(t_ps, (states["x_e"] + states["x_h"]) / 1e6, label=label)
axes[0].set(ylabel="Load current (mA)")
axes[1].set(ylabel="Effective internal field (MV/m)")
axes[2].set(ylabel="Total excess concentration (cm⁻³)")
for ax in axes:
    ax.set(xlabel="Time (ps)", xlim=(60, 1200))
    ax.legend()
plt.tight_layout()
plt.show()

screening_ratio = on_states["field_screened"] / on_states["field_unscreened"]
assert np.min(screening_ratio) < 0.5
np.testing.assert_allclose(off_states["field_screened"], off_states["field_unscreened"])
assert np.max(feedback_on["carrier"]) < np.max(feedback_off["carrier"])
print(f"Minimum screened / unscreened internal field: {np.min(screening_ratio):.3f}")
for label, current_set in [("off", feedback_off), ("on", feedback_on)]:
    peak, fraction, rise, fall = pulse_metrics(current_set["carrier"], 1e-2)
    print(f"Screening {label}: peak={peak * 1e3:.3f} mA; charge fraction={fraction:.4f}; rise/fall={rise:.1f}/{fall:.1f} ps")

# Show rise/fall intervals directly, using the same symmetric optical stimulus.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
shape_metrics = {}
optical_shape = np.exp(-0.5 * ((np.asarray(times) - 120e-12) / 12e-12)**2)
for ax, label, current_set in zip(axes, ("screening off", "screening on"),
                                 (feedback_off, feedback_on), strict=True):
    current = current_set["carrier"]
    k = int(np.argmax(current))
    normalized = current / current[k]
    rise10 = crossing_time(t_ps[:k + 1], normalized[:k + 1], 0.1)
    rise90 = crossing_time(t_ps[:k + 1], normalized[:k + 1], 0.9)
    fall90 = crossing_time(t_ps[k:], normalized[k:], 0.9)
    fall10 = crossing_time(t_ps[k:], normalized[k:], 0.1)
    rise, fall = rise90 - rise10, fall10 - fall90
    shape_metrics[label] = rise, fall
    ax.plot(t_ps, normalized, color="0.15", lw=2, label="normalized photocurrent")
    ax.plot(t_ps, optical_shape, "--", color="0.6", label="normalized optical power")
    ax.axvspan(rise10, rise90, alpha=0.2, color="C0", label=f"10–90% rise: {rise:.1f} ps")
    ax.axvspan(fall90, fall10, alpha=0.2, color="C1", label=f"90–10% fall: {fall:.1f} ps")
    ax.scatter([rise10, rise90, fall90, fall10], [0.1, 0.9, 0.9, 0.1], color="0.15", s=24, zorder=3)
    ax.set(title=f"{label}: fall / rise = {fall / rise:.1f}", xlabel="Time (ps)",
           ylabel="Normalized power or current", xlim=(60, 650), ylim=(-0.03, 1.08))
    ax.legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

rise_off, fall_off = shape_metrics["screening off"]
rise_on, fall_on = shape_metrics["screening on"]
assert fall_on > 3 * fall_off
assert fall_on / rise_on > fall_off / rise_off
print(f"Fall/rise ratio: {fall_off / rise_off:.2f} without screening, {fall_on / rise_on:.2f} with screening")


## Conservation and numerical checks

First: screening and recombination both off, confirming that unequal sweep-out times redistribute the impulse without inventing or losing charge. Then: half-size time step for convergence, phase independence, and the power sweep from above. Baseline subtraction strips the static dark current, so integrated load charge should match external responsivity × incident pulse energy — the junction charge returns to its starting value at the integration endpoints.


In [ ]:
linear_currents, linear_densities = simulate(power=1e-5, screening=0.0, tau_rec=1e6)
linear_metrics = pulse_metrics(linear_currents["carrier"], 1e-5)
np.testing.assert_allclose(linear_metrics[1], 1.0, rtol=0.01)
np.testing.assert_allclose(pulse_metrics(currents["ideal"], 1e-3)[1], 1.0, rtol=0.002)
np.testing.assert_allclose(pulse_metrics(currents["diode"], 1e-3)[1], 1.0, rtol=0.005)

refined, _ = simulate(dt=0.5e-12)
relative_error = np.max(np.abs(refined["carrier"] - currents["carrier"])) / np.max(refined["carrier"])
assert relative_error < 0.01, relative_error

low_peak, low_charge, *_ = pulse_metrics(sweep[1e-5][0]["carrier"], 1e-5)
high_peak, high_charge, high_rise, high_fall = pulse_metrics(sweep[1e-2][0]["carrier"], 1e-2)
assert high_peak / 1e-2 < low_peak / 1e-5  # peak-current compression
assert 0 < high_charge < low_charge <= 1.01  # recombination reduces collected charge
assert high_fall > high_rise
assert np.max(sweep[1e-2][1]["x_h"][-1:]) < 1e17  # return near dark population (m^-3)

# Phase independence measured through the complete transient testbench.
phase_currents, phase_densities = simulate(phase=1.2)
for name in detector_names:
    np.testing.assert_allclose(phase_currents[name], currents[name], rtol=1e-6, atol=1e-10)
for state in ("x_e", "x_h"):
    np.testing.assert_allclose(phase_densities[state], concentrations[state], rtol=1e-6, atol=1e14)
assert min(np.min(density) for _, densities in sweep.values() for state, density in densities.items() if state in ("x_e", "x_h")) > -1e16
print(f"Linear collected charge: {linear_metrics[1]:.6f} × external responsivity × incident pulse energy")
print(f"Half-step peak-relative waveform error: {relative_error:.3%}")
print(f"High-power rise/fall: {high_rise:.2f} / {high_fall:.2f} ps")
print("All notebook checks passed.")

## Spectral response and reflected waves

Sweep `wavelength_nm` across all three models at low power, with the Gaussian source centered at t=0 so we get an illuminated DC operating point. All three sources still preserve the same incident power at every wavelength.

We plot absorbed power vs. external responsivity, power reflectance, and reflected-field phase. The solved optical field must satisfy $E=(1+r)a$, so pulling $b=E-a$ out of the circuit checks both reflection magnitude and phase against the analytic profile. We also check $P_\mathrm{inc}=P_\mathrm{refl}+P_\mathrm{abs}$ and that the ideal detector's photocurrent tracks the spectral profile exactly. The diode models differ slightly at low power — that's dark-current subtraction and recombination showing through.


In [ ]:
wavelengths_nm = jnp.linspace(1200, 1800, 121)
spectral_power = 1e-6
r_abs, reflection = spectral_response(wavelength_nm=wavelengths_nm)
external_r = r_abs * (1 - jnp.abs(reflection)**2)
dark_op = circuit.dc(params={"power": 0.0}, rtol=1e-10, atol=1e-13)
spectral_op = circuit.dc(params={"power": spectral_power, "center": 0.0,
                                 "wavelength_nm": wavelengths_nm}, rtol=1e-10, atol=1e-13)
measured_r = {
    name: np.real((circuit.port(spectral_op, name) - circuit.port(spectral_op, "GND,p1"))
                  - (circuit.port(dark_op, name) - circuit.port(dark_op, "GND,p1"))) / (50 * spectral_power)
    for name in ("ideal", "diode", "carrier")
}
incident = np.sqrt(spectral_power)
field = circuit.port(spectral_op, "light")
reflected = field - incident
np.testing.assert_allclose(reflected / incident, reflection, rtol=1e-7, atol=1e-9)
np.testing.assert_allclose(measured_r["ideal"], external_r, rtol=1e-7, atol=1e-9)

absorbed = spectral_power - jnp.abs(reflected)**2
np.testing.assert_allclose(absorbed + jnp.abs(reflected)**2, spectral_power, rtol=1e-7)
np.testing.assert_allclose(measured_r["ideal"] * spectral_power, r_abs * absorbed, rtol=1e-7)
assert bool(jnp.all(absorbed >= 0))
# Physical upper bound for the illustrative absorbed-power conversion efficiency.
quantum_limit = QE * wavelengths_nm * 1e-9 / (6.62607015e-34 * 299792458.0)
assert bool(jnp.all((r_abs >= 0) & (r_abs <= quantum_limit)))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(wavelengths_nm, r_abs, "--", color="0.4", label="absorbed-power profile")
for name, response in measured_r.items():
    axes[0].plot(wavelengths_nm, response, label=f"external: {name}",
                 linestyle={"ideal": "-", "diode": "--", "carrier": ":"}[name])
axes[0].set(ylabel="Responsivity (A/W)")
axes[1].plot(wavelengths_nm, jnp.abs(reflection)**2 * 100, label="profile")
axes[1].plot(wavelengths_nm[::8], np.asarray(jnp.abs(reflected / incident)**2)[::8] * 100,
             "o", ms=4, label="circuit")
axes[1].set(ylabel="Power reflectance (%)")
axes[2].plot(wavelengths_nm, np.unwrap(np.angle(reflection)), label="profile")
axes[2].plot(wavelengths_nm[::8], np.unwrap(np.angle(reflected / incident))[::8],
             "o", ms=4, label="circuit")
axes[2].set(ylabel="Unwrapped reflection phase (rad)")
for ax in axes:
    ax.set(xlabel="Optical carrier wavelength (nm)")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Matched limit and differentiability through a complete DC solve.
matched_op = circuit.dc(params={"power": spectral_power, "center": 0.0, "reflectivity": 0.0})
np.testing.assert_allclose(circuit.port(matched_op, "light"), jnp.sqrt(spectral_power), rtol=1e-7)
np.testing.assert_allclose(jnp.real(circuit.port(matched_op, "ideal")) / 50,
                           RESPONSIVITY * spectral_power, rtol=1e-6)


def solved_photocurrent(wavelength):
    op = circuit.dc(params={"power": spectral_power, "center": 0.0, "wavelength_nm": wavelength})
    return jnp.real(circuit.port(op, "ideal") - circuit.port(op, "GND,p1")) / 50


spectral_gradient = jax.grad(solved_photocurrent)(1500.0)
assert bool(jnp.isfinite(spectral_gradient))
assert abs(float(spectral_gradient)) > 1e-12
print("Solver spectral responsivity, complex reflection, energy balance, and wavelength-gradient checks passed.")


## Bias-dependent S-parameters and AC bandwidth

Same device profiles, small CW optical power (1 µW), fixed `wavelength_nm=1550`, sweeping modulation frequency in Hz. Sources are centered at t=0 so their DC stamp directly gives the CW operating point to linearize around.

Reverse bias shrinks depletion capacitance. With the 50 Ω load, the predicted electrical pole is

$$
f_\mathrm{RC}=\frac{1/R_L+g_\mathrm{dark}}{2\pi C(V_\mathrm{DC})},\qquad
g_\mathrm{dark}=dI_\mathrm{dark}/dV.
$$

First, `circuit.sp()` sweeps the diode's electrical port. Converting S11 back to impedance via $Z_\mathrm{in}=Z_0(1+S_{11})/(1-S_{11})$ and normalizing gives the response to a small injected current, isolating the C-V-dependent RC bandwidth — this removes the S-parameter solver's reference-port termination, the actual 50 Ω load stays in the circuit.

Then `device.sp()` solves the full linearized circuit for a small optical power modulation to get $\delta I_\mathrm{load}/\delta P_\mathrm{inc}$ in A/W. The diode adds electrical RC filtering on top; the carrier model adds transport and screening feedback around the operating point. Because intensity and screening aren't holomorphic, `holomorphic=False` selects Circulax's full real/imaginary block solver. A six-port SP measurement probes the three optical nodes and three load nodes. The standard $Z=(I+S)(I-S)^{-1}$ conversion at a uniform 1 Ω reference removes the added SP reference terminations; the actual matched sources and loads remain. At the reference wavelength and zero phase, amplitude and phase quadratures decouple, so this port impedance gives the power-modulation response. Optical conversion gain (A/W) is plotted separately from the dimensionless S11.

At these illustrative 20 fF parameters, carrier transport limits optical bandwidth well before the electrical RC pole does. Bump `c0` in `ac_device()` if you want the RC pole to dominate instead.

`optical_ac_sweep()` vmaps the DC and SP solves together across bias or power. Frequency is swept inside `sp()`. Arrays passed to both bias and power describe paired points; a scalar broadcasts over the other axis. Only plotting and tabular reporting use Python loops.


In [ ]:
from circulax.utils import apply_global_params

ac_freqs = jnp.concatenate([jnp.array([0.0]), jnp.logspace(7, 12, 241)])
reverse_biases = [0.5, 1.0, 3.0, 6.0]
ac_power = 1e-6


def ac_device(bias, c0=20e-15, power=ac_power, screening=1.0, tau_rec=500e-12, tau_e0=8e-12, tau_h0=40e-12, base_circuit=circuit):
    groups = apply_global_params(base_circuit.groups, {
        "V": bias, "power": power, "center": 0.0, "screening": screening, "tau_rec": tau_rec,
        "wavelength_nm": REFERENCE_WAVELENGTH_NM, "c0": c0, "tau_e0": tau_e0, "tau_h0": tau_h0})
    device = base_circuit.with_groups(groups)
    op = device.dc(rtol=1e-10, atol=1e-13)
    return device, op


def optical_ac_response(device, op, frequencies, power=ac_power, params=None):
    # Public SP solver performs all circuit linearization and frequency solves.
    # At the reference wavelength and zero optical phase the amplitude and
    # phase quadratures decouple. This measures amplitude/power modulation.
    ports = [f"LIGHT_{name},p1" for name in detector_names] + list(detector_names)
    scattering = device.sp(ports=ports, freqs=frequencies, z0=1.0,
                           y_dc=op, holomorphic=False, params=params)
    # Standard network postprocessing removes only SP reference terminations.
    # The matched optical sources and physical 50-ohm loads stay in the circuit.
    # Z = (I + S) (I - S)^-1 at the uniform 1-ohm reference.
    # Use a right matrix solve rather than an explicit inverse. Keeping this
    # standard port conversion in JAX lets the entire measurement be vmapped.
    identity = jnp.eye(len(ports))
    impedance = jnp.linalg.solve(
        (identity - scattering).swapaxes(-1, -2),
        (identity + scattering).swapaxes(-1, -2),
    ).swapaxes(-1, -2)
    # Source I = E - 2 sqrt(P) gives a differential injection dP/sqrt(P).
    # Sum the three optical injections; return load current per incident watt.
    return {name: impedance[:, 3 + k, :3].sum(axis=1) / (50 * jnp.sqrt(power))
            for k, name in enumerate(detector_names)}


def optical_ac_sweep(base_circuit, biases, powers, *, include_electrical=False, **settings):
    # Scalars broadcast; two arrays describe paired operating points.
    biases, powers = jnp.broadcast_arrays(jnp.atleast_1d(biases), jnp.atleast_1d(powers))

    def solve_point(bias, power):
        device, op = ac_device(bias, power=power, base_circuit=base_circuit, **settings)
        response = optical_ac_response(device, op, ac_freqs, power=power)
        s11 = (device.sp(ports="diode", freqs=ac_freqs, z0=50.0,
                         y_dc=op, holomorphic=False)[:, 0, 0] if include_electrical else None)
        return {"op": op, "response": response, "s11": s11}

    return jax.vmap(solve_point)(biases, powers)


def bandwidth_3db(frequencies, normalized_response):
    f = np.asarray(frequencies)[1:]
    db = 20 * np.log10(np.abs(normalized_response)[1:])
    target = -10 * np.log10(2)
    crossings = np.flatnonzero((db[:-1] > target) & (db[1:] <= target))
    if not len(crossings):
        return np.nan
    k = crossings[0]
    log_f = np.log10(f[k]) + (target - db[k]) * (np.log10(f[k + 1]) - np.log10(f[k])) / (db[k + 1] - db[k])
    return 10**log_f


fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
print("Reverse bias  C at op (fF)  RC predicted (GHz)  RC AC (GHz)  Optical carrier (GHz)")
bias_ac_sweep = optical_ac_sweep(circuit, jnp.array(reverse_biases), ac_power, include_electrical=True)
ac_results = {}
for k, bias in enumerate(reverse_biases):
    op = bias_ac_sweep["op"][k]
    device = circuit
    v_op = float(jnp.real(device.port(op, "diode") - device.port(op, "BIAS,p1")))
    c_op = float(jax.grad(junction_charge)(v_op))
    g_op = 1 / 50 + float(jax.grad(dark_iv)(v_op))
    predicted_rc = g_op / (2 * np.pi * c_op)
    s11 = np.asarray(bias_ac_sweep["s11"][k])
    impedance = 50 * (1 + s11) / (1 - s11)
    rc_response = impedance / impedance[0]
    analytic_rc = g_op / (g_op + 2j * np.pi * np.asarray(ac_freqs) * c_op)
    np.testing.assert_allclose(rc_response, analytic_rc, rtol=2e-5, atol=2e-7)
    measured_rc = bandwidth_3db(ac_freqs, rc_response)
    np.testing.assert_allclose(measured_rc, predicted_rc, rtol=0.01)

    optical_response = jax.tree.map(lambda values, k=k: values[k], bias_ac_sweep["response"])
    diode_normalized = np.asarray(optical_response["diode"] / optical_response["diode"][0])
    np.testing.assert_allclose(diode_normalized, analytic_rc, rtol=2e-5, atol=2e-7)
    carrier_normalized = np.asarray(optical_response["carrier"] / optical_response["carrier"][0])
    carrier_bw = bandwidth_3db(ac_freqs, carrier_normalized)
    assert np.isfinite(carrier_bw) and carrier_bw < measured_rc
    np.testing.assert_allclose(optical_response["ideal"], EXTERNAL_RESPONSIVITY, rtol=1e-5)
    ac_results[bias] = {"capacitance": c_op, "rc_bandwidth": measured_rc,
                        "carrier_bandwidth": carrier_bw, "optical_response": optical_response}
    print(f"{bias:12.1f} {c_op * 1e15:13.2f} {predicted_rc / 1e9:19.2f} {measured_rc / 1e9:12.2f} {carrier_bw / 1e9:22.2f}")
    ghz = np.asarray(ac_freqs)[1:] / 1e9
    color = f"C{k}"
    axes[0].semilogx(ghz, 20 * np.log10(np.maximum(np.abs(s11[1:]), 1e-14)), color=color, label=f"{bias:g} V")
    axes[1].semilogx(ghz, 20 * np.log10(np.abs(rc_response[1:])), color=color, label=f"{bias:g} V, {c_op * 1e15:.1f} fF")
    axes[2].semilogx(ghz, 20 * np.log10(np.abs(diode_normalized[1:])), color=color, label=f"{bias:g} V diode")
    axes[2].semilogx(ghz, 20 * np.log10(np.abs(carrier_normalized[1:])), "--", color=color, label=f"{bias:g} V carriers")
axes[0].set(ylabel="Electrical |S11| (dB)", title="Diode electrical reflection")
axes[1].set(ylabel="Normalized transimpedance (dB)", title="C–V-dependent RC bandwidth", ylim=(-25, 1))
axes[2].set(ylabel="Normalized optical conversion gain (dB)", title="RC and carrier transport", ylim=(-40, 1))
for ax in axes:
    ax.set(xlabel="Modulation frequency (GHz)")
    ax.legend(fontsize=7, loc="lower left")
for ax in axes[1:]:
    ax.axhline(-10 * np.log10(2), color="0.5", linestyle=":", linewidth=1)
plt.tight_layout()
plt.show()

assert np.all(np.diff([ac_results[b]["capacitance"] for b in reverse_biases]) < 0)
assert np.all(np.diff([ac_results[b]["rc_bandwidth"] for b in reverse_biases]) > 0)
assert np.all(np.diff([ac_results[b]["carrier_bandwidth"] for b in reverse_biases]) > 0)
print("Bias-dependent S11, RC pole, and optical AC bandwidth checks passed.")


## Screening-dependent AC response at nominal -2 V

Supply held at 2 V reverse bias, sweeping CW incident power from 1 µW to 0.5 mW at 1550 nm. We solve a fresh illuminated operating point per power level and apply infinitesimal fractional modulation — these are local AC responses around a nonlinear equilibrium, not Fourier transforms of a large pulse. The table's junction voltage includes the DC drop across the 50 Ω load.

`tau_rec=1e6` s **in this experiment only** — makes recombination negligible so we isolate collection-field screening cleanly. We stay below the model's steady-state collection limit: without recombination, pushing generation higher would just pile up carriers forever instead of settling to equilibrium. The pulse/saturation examples above keep their normal recombination lifetime.

This is also a separate **fast-detector scenario**: `tau_e0=1.2` ps, `tau_h0=6` ps, screening strength 1.5 — tuned for ~30 GHz low-power bandwidth with roughly a 10 GHz drop across the power sweep. Not fit to any real device. The earlier pulse examples kept slower sweep-out times so the electron/hole tails were easy to see by eye. At -2 V the bias-dependent sweep-out times are longer than the reference times (specified at -3 V).

Left panel: original 20 fF junction, 50 Ω load. As power → 0, the carrier model converges to the unscreened transport response (fixed sweep-out times still filter the signal). Higher power → weaker field → longer sweep-out → lower bandwidth.

Right panel: same thing plus **10 pF shunt load capacitance** — junction, volume, sweep-out times, and load R all unchanged. Its RC pole is slower than the low-power collection bandwidth, so the low-power carrier curve converges onto the diode curve through the RC passband, and this fixture stays mostly RC-limited across the whole power range. The transport-driven bandwidth reduction is clearest in the left panel — a slow enough load can hide it. The capacitance here is an explicit test-fixture/load choice, not a change to the absorber itself.

Both panels normalize to the same low-power external responsivity (0.768 A/W). Full collection keeps the DC gain fixed even as screening reshapes the frequency response. Gray curve: screening forced off at 0.5 mW, everything else fixed. Black dashed: diode at the lowest power.

The table ties internal-field suppression and separate electron/hole sweep-out times to -3 dB bandwidth — screening lowers it in this isolated-transport setup. With strong recombination the picture isn't necessarily monotonic though: gain suppression and shifting relaxation modes can give you a wider-looking but much weaker response normalized to its own DC gain. Read the earlier finite-recombination results together with their absolute gain and collected charge, not the normalized bandwidth alone.


In [ ]:
from copy import deepcopy
from circulax.components.electronic import Capacitor

# A separate, explicitly loaded fixture makes the low-power diode limit visible.
loaded_netlist = deepcopy(netlist)
for name in ("ideal", "diode", "carrier"):
    loaded_netlist["instances"][f"C_load_{name}"] = {
        "component": "load_capacitor", "settings": {"C": 10e-12}}
    loaded_netlist["nets"].extend([
        {"p1": f"PD_{name},anode", "p2": f"C_load_{name},p1"},
        {"p1": "GND,p1", "p2": f"C_load_{name},p2"},
    ])
loaded_circuit = compile_circuit(
    loaded_netlist, {**models, "load_capacitor": Capacitor},
    is_complex=True, backend="dense", rtol=1e-8, atol=1e-10)

power_levels = np.array([1e-6, 1e-5, 1e-4, 3e-4, 5e-4])
transport_tau_rec = 1e6  # negligible recombination, matching the screening-only pulse experiment
fixed_reverse_supply = 2.0
# A separate fast-detector scenario: approximately 30 GHz at low power,
# with an illustrative 10 GHz screening-induced drop across this power range.
transport_settings = {"tau_e0": 1.2e-12, "tau_h0": 6e-12, "screening": 1.5}
power_ac_results = {}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
print("Load C (pF)  P (mW)  Junction (V)  Field / E0  tau_e (ps)  tau_h (ps)  Gain (A/W)  BW (GHz)")

for row, (label, fixture, load_cap_pf) in enumerate([
    ("Fast-detector scenario: 50 Ω load", circuit, 0.0),
    ("50 Ω load + 10 pF fixture capacitance", loaded_circuit, 10.0),
]):
    fixture_sweep = optical_ac_sweep(
        fixture, fixed_reverse_supply, jnp.asarray(power_levels),
        tau_rec=transport_tau_rec, **transport_settings,
    )
    results = {}
    for k, power in enumerate(power_levels):
        device = fixture
        op = fixture_sweep["op"][k]
        response = jax.tree.map(lambda values, k=k: values[k], fixture_sweep["response"])
        carrier = np.asarray(response["carrier"])
        normalized = carrier / carrier[0]
        v = device.port(op, "carrier") - device.port(op, "BIAS,p1")
        x_e = device.port(op, "PD_carrier,x_e")
        x_h = device.port(op, "PD_carrier,x_h")
        e0, field = internal_field(x_e, x_h, v, screening=transport_settings["screening"])
        tau_e, tau_h, _ = carrier_rates(x_e, x_h, v,
            tau_e0=transport_settings["tau_e0"], tau_h0=transport_settings["tau_h0"],
            screening=transport_settings["screening"], tau_rec=transport_tau_rec)
        gain = float(np.real(carrier[0]))
        bw = bandwidth_3db(ac_freqs, normalized)
        assert gain > 0 and np.isfinite(bw)
        results[power] = {"response": response, "gain": gain, "bandwidth": bw,
                          "field_ratio": float(field / e0), "junction_v": float(jnp.real(v))}
        print(f"{load_cap_pf:11.0f} {power * 1e3:8.3g} {float(jnp.real(v)):13.4f} "
              f"{float(field / e0):11.3f} {float(tau_e) * 1e12:11.2f} {float(tau_h) * 1e12:11.2f} "
              f"{gain:11.4f} {bw / 1e9:9.3f}")
        ghz = np.asarray(ac_freqs)[1:] / 1e9
        axes[row].semilogx(ghz, 20 * np.log10(np.abs(carrier[1:]) / EXTERNAL_RESPONSIVITY),
                           color=f"C{k}", label=f"{power * 1e3:g} mW")

    # Compare the identical high-power circuit with only screening disabled.
    high_power = power_levels[-1]
    no_screen_device, no_screen_op = ac_device(
        fixed_reverse_supply, power=high_power, tau_rec=transport_tau_rec, base_circuit=fixture,
        **{**transport_settings, "screening": 0.0})
    no_screen_response = optical_ac_response(no_screen_device, no_screen_op, ac_freqs, power=high_power)
    no_screen_carrier = np.asarray(no_screen_response["carrier"])
    diode = np.asarray(results[power_levels[0]]["response"]["diode"])
    axes[row].semilogx(ghz, 20 * np.log10(np.abs(diode[1:]) / EXTERNAL_RESPONSIVITY), "--",
                       color="black", lw=2, label="low-power diode reference")
    axes[row].semilogx(ghz, 20 * np.log10(np.abs(no_screen_carrier[1:]) / EXTERNAL_RESPONSIVITY), ":",
                       color="0.4", lw=2, label="0.5 mW, screening off")
    axes[row].set(title=label, ylim=(-40, 1), xlim=(0.1, 500) if row == 0 else (0.01, 20),
                  ylabel="Differential gain / external responsivity (dB)", xlabel="Modulation frequency (GHz)")
    axes[row].legend(fontsize=8, loc="lower left")
    axes[row].axhline(-10 * np.log10(2), color="0.5", ls=":", lw=1)

    assert results[high_power]["field_ratio"] < results[power_levels[0]]["field_ratio"]
    np.testing.assert_allclose([results[p]["gain"] for p in power_levels], EXTERNAL_RESPONSIVITY, rtol=1e-4)
    assert np.all(np.diff([results[p]["bandwidth"] for p in power_levels]) < 0)
    assert results[high_power]["bandwidth"] < bandwidth_3db(
        ac_freqs, no_screen_carrier / no_screen_carrier[0])
    power_ac_results[load_cap_pf] = results

plt.tight_layout()
plt.show()

# Quantify the low-power convergence in the intentionally RC-limited fixture.
loaded_results = power_ac_results[10.0]
low_response = loaded_results[power_levels[0]]["response"]
low_carrier = np.asarray(low_response["carrier"] / low_response["carrier"][0])
low_diode = np.asarray(low_response["diode"] / low_response["diode"][0])
diode_bandwidth = bandwidth_3db(ac_freqs, low_diode)
passband = np.asarray(ac_freqs) <= diode_bandwidth
low_power_difference_db = np.max(np.abs(20 * np.log10(np.abs(low_carrier[passband] / low_diode[passband]))))
assert low_power_difference_db < 0.1
np.testing.assert_allclose(loaded_results[power_levels[0]]["bandwidth"], diode_bandwidth, rtol=0.02)
assert loaded_results[power_levels[-1]]["bandwidth"] < loaded_results[power_levels[0]]["bandwidth"]
print(f"RC-limited low-power carrier/diode difference through the diode passband: {low_power_difference_db:.3f} dB")
print(f"RC-limited diode bandwidth: {diode_bandwidth / 1e9:.3f} GHz")
print("Fixed-bias power sweep, low-power diode limit, and screening-control checks passed.")

low_bw = power_ac_results[0.0][power_levels[0]]["bandwidth"]
high_bw = power_ac_results[0.0][power_levels[-1]]["bandwidth"]
bandwidth_drop_ghz = (low_bw - high_bw) / 1e9
print(f"Illustrative fast-detector bandwidth: {low_bw / 1e9:.2f} → {high_bw / 1e9:.2f} GHz")
print(f"Screening-induced reduction across the power sweep: {bandwidth_drop_ghz:.2f} GHz")
assert 8 < bandwidth_drop_ghz < 12


## With and without pad/metal parasitics

Building a reusable photodetector subcircuit and dropping it into the full testbench. The package wraps the intrinsic detector with a true series resistor and a parallel pad R-C network: **20 Ω series resistance**, **100 fF pad capacitance**, **1 MΩ pad shunt resistance** — all configurable. The pad network goes from the external output to its own ground; the 50 Ω measurement load is separate from that.

```text
intrinsic PD anode ── R_series ── output ── R_load (50 Ω) ── ground
                                  ├────── R_pad ─────────── ground
                                  └────── C_pad ─────────── ground
PD cathode ── +2 V bias supply
```

`make_photodetector_subcircuit()` compiles a `Circuit` with ports `optical_in`, `anode`, `cathode`, `pad_ground`. Pad ground is kept separate so pad capacitance references ground rather than the reverse-biased cathode. `make_parasitic_fixture()` drops these compiled packages straight into the models mapping of the full testbench — Circulax expands the hierarchy at compile time, and the original device-only circuit is untouched. The I-V/C-V/carrier equations still see the intrinsic anode-to-cathode voltage, upstream of the series resistor — the junction anode becomes an internal node (`PD_carrier~CORE,anode` once expanded), only reaching the external anode through `R_series`. The checks below confirm the resistor actually sits between two distinct nodes (not bypassed). Named output ports map to the external load node, so AC gain measures current delivered to the 50 Ω load, not current leaking into the pad.

Same power sweep and fast-detector settings as the previous section. The frequency plot compares lowest/highest power, dashed = device only, solid = with parasitics. Bandwidth plot covers the whole power range plus a diode-with-parasitics reference. A screening-off control separates transport feedback from pure electrical filtering. Toggling just `C_pad` off checks that it changes AC response without touching the DC point.

DC gain division from pad leakage is $R_\mathrm{pad}/(R_\mathrm{pad}+50\,\Omega)$. Series drop and pad capacitance add their own frequency dependence on top, and the full circuit solve captures how that interacts with junction capacitance and carrier storage. These numbers, and the ~10 GHz device-only bandwidth target, are illustrative — not calibrated to a real package.

`with_parasitics=False` gives you the bare package — just the intrinsic core, original three ports, no resistor or pad branches. Checks at both ends of the power sweep confirm this bare package matches the original intrinsic-component circuit exactly.


In [ ]:
def make_photodetector_subcircuit(model_kind="carrier", series_resistance=20.0,
                                  pad_resistance=1e6, pad_capacitance=100e-15, with_parasitics=True):
    """Reusable PD package: intrinsic core, true series R, and pad R || C.

    Physical ports: optical_in, anode (external), cathode, and pad_ground.
    The core's junction anode is an internal node on the other side of R_series.
    """
    if model_kind not in ("ideal", "diode", "carrier"):
        raise ValueError("model_kind must be ideal, diode, or carrier")
    if not with_parasitics:
        return compile_circuit({
            "instances": {"CORE": {"component": model_kind}}, "connections": {},
            "ports": {port: f"CORE,{port}" for port in PORTS},
        }, models, is_complex=True, backend="dense", rtol=1e-8, atol=1e-10)
    package_netlist = {
        "instances": {
            "CORE": {"component": model_kind},
            "R_series": {"component": "load", "settings": {"R": series_resistance}},
            "R_pad": {"component": "load", "settings": {"R": pad_resistance}},
        },
        "connections": {},
        "nets": [
            {"p1": "CORE,anode", "p2": "R_series,p1"},
            {"p1": "R_series,p2", "p2": "R_pad,p1"},
        ],
        "ports": {
            "optical_in": "CORE,optical_in",
            "anode": "R_series,p2",
            "cathode": "CORE,cathode",
            "pad_ground": "R_pad,p2",
        },
    }
    if pad_capacitance > 0:
        package_netlist["instances"]["C_pad"] = {
            "component": "pad_capacitor", "settings": {"C": pad_capacitance}}
        package_netlist["nets"].extend([
            {"p1": "R_series,p2", "p2": "C_pad,p1"},
            {"p1": "R_pad,p2", "p2": "C_pad,p2"},
        ])
    return compile_circuit(package_netlist, {**models, "pad_capacitor": Capacitor},
                           is_complex=True, backend="dense", rtol=1e-8, atol=1e-10)


def make_parasitic_fixture(series_resistance=20.0, pad_resistance=1e6, pad_capacitance=100e-15, with_parasitics=True):
    # The full circuit contains PD package instances plus sources and loads.
    # Circulax accepts compiled Circuit objects directly in the models mapping.
    packaged_device = deepcopy(device_netlist)
    fixture_models = dict(models)
    for name in ("ideal", "diode", "carrier"):
        package_name = f"packaged_{name}"
        fixture_models[package_name] = make_photodetector_subcircuit(
            name, series_resistance, pad_resistance, pad_capacitance, with_parasitics=with_parasitics)
        packaged_device["instances"][f"PD_{name}"] = {"component": package_name}
        if with_parasitics:
            packaged_device["ports"][f"pad_ground_{name}"] = f"PD_{name},pad_ground"
        # Existing optical, bias, anode-load, and output-port connections now
        # address the package's physical ports, rather than individual parts.
    fixture_netlist = make_testbench(packaged_device)
    return compile_circuit(fixture_netlist, fixture_models,
                           is_complex=True, backend="dense", rtol=1e-8, atol=1e-10)


series_resistance = 20.0
pad_resistance = 1e6
pad_capacitance = 100e-15
parasitic_circuit = make_parasitic_fixture(series_resistance, pad_resistance, pad_capacitance)
# Verify topology: R_series connects two distinct nodes, with no bypass.
port_map = parasitic_circuit.port_map
assert port_map["PD_carrier~CORE,anode"] == port_map["PD_carrier~R_series,p1"]
assert port_map["carrier"] == port_map["PD_carrier~R_series,p2"]
assert port_map["carrier"] == port_map["PD_carrier~R_pad,p1"] == port_map["PD_carrier~C_pad,p1"]
assert port_map["PD_carrier~CORE,anode"] != port_map["carrier"]
parasitic_ac_results = {}
print("P (mW)  Device BW (GHz)  With parasitics (GHz)  Diode+parasitics (GHz)  Junction (V)  Series drop (mV)")
parasitic_sweep = optical_ac_sweep(
    parasitic_circuit, fixed_reverse_supply, jnp.asarray(power_levels),
    tau_rec=transport_tau_rec, **transport_settings,
)
for k, power in enumerate(power_levels):
    device = parasitic_circuit
    op = parasitic_sweep["op"][k]
    response = jax.tree.map(lambda values, k=k: values[k], parasitic_sweep["response"])
    carrier = np.asarray(response["carrier"])
    diode = np.asarray(response["diode"])
    carrier_bw = bandwidth_3db(ac_freqs, carrier / carrier[0])
    diode_bw = bandwidth_3db(ac_freqs, diode / diode[0])
    junction_v = device.port(op, "PD_carrier~CORE,anode") - device.port(op, "BIAS,p1")
    external_v = device.port(op, "carrier") - device.port(op, "GND,p1")
    series_drop = device.port(op, "PD_carrier~CORE,anode") - device.port(op, "carrier")
    expected_series_drop = series_resistance * external_v * (1 / 50 + 1 / pad_resistance)
    np.testing.assert_allclose(series_drop, expected_series_drop, rtol=1e-7, atol=1e-12)
    expected_load_gain = EXTERNAL_RESPONSIVITY * pad_resistance / (pad_resistance + 50)
    np.testing.assert_allclose(carrier[0], expected_load_gain, rtol=1e-4)
    bare_bw = power_ac_results[0.0][power]["bandwidth"]
    assert 0 < carrier_bw < bare_bw
    parasitic_ac_results[power] = {"response": response, "bandwidth": carrier_bw,
                                  "diode_bandwidth": diode_bw, "op": op, "device": device}
    print(f"{power * 1e3:7.3g} {bare_bw / 1e9:16.2f} {carrier_bw / 1e9:22.2f} "
          f"{diode_bw / 1e9:23.2f} {float(jnp.real(junction_v)):13.4f} {float(jnp.real(series_drop)) * 1e3:17.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ghz = np.asarray(ac_freqs)[1:] / 1e9
for color, power in [("C0", power_levels[0]), ("C4", power_levels[-1])]:
    for label, response, linestyle in [
        ("without", power_ac_results[0.0][power]["response"]["carrier"], "--"),
        ("with", parasitic_ac_results[power]["response"]["carrier"], "-"),
    ]:
        axes[0].semilogx(ghz, 20 * np.log10(np.abs(response[1:]) / EXTERNAL_RESPONSIVITY),
                         linestyle=linestyle, color=color, lw=2,
                         label=f"{power * 1e3:g} mW, {label} parasitics")

no_screen_device, no_screen_op = ac_device(
    fixed_reverse_supply, power=power_levels[-1], tau_rec=transport_tau_rec,
    base_circuit=parasitic_circuit, **{**transport_settings, "screening": 0.0})
no_screen = optical_ac_response(no_screen_device, no_screen_op, ac_freqs, power=power_levels[-1])["carrier"]
assert bandwidth_3db(ac_freqs, no_screen / no_screen[0]) > parasitic_ac_results[power_levels[-1]]["bandwidth"]
axes[0].semilogx(ghz, 20 * np.log10(np.abs(no_screen[1:]) / EXTERNAL_RESPONSIVITY), ":",
                 color="0.4", label="0.5 mW, parasitics on, screening off")
axes[0].set(xlabel="Modulation frequency (GHz)", ylabel="Gain / external responsivity (dB)",
            title="Transport and parasitics at nominal −2 V", xlim=(0.1, 200), ylim=(-35, 1))
axes[0].axhline(-10 * np.log10(2), color="0.5", ls=":", lw=1)
axes[0].legend(fontsize=7, loc="lower left")

powers_mw = power_levels * 1e3
axes[1].semilogx(powers_mw, [power_ac_results[0.0][p]["bandwidth"] / 1e9 for p in power_levels],
                 "o--", label="without parasitics")
axes[1].semilogx(powers_mw, [parasitic_ac_results[p]["bandwidth"] / 1e9 for p in power_levels],
                 "o-", label="with parasitics")
axes[1].semilogx(powers_mw, [parasitic_ac_results[p]["diode_bandwidth"] / 1e9 for p in power_levels],
                 ":", color="0.3", label="diode with parasitics")
axes[1].set(xlabel="CW incident power (mW)", ylabel="−3 dB bandwidth (GHz)",
            title="Power sweep with the same intrinsic device")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

# C_pad is reactive: removing just that branch leaves the DC solution intact.
no_cap_circuit = make_parasitic_fixture(series_resistance, pad_resistance, pad_capacitance=0.0)
no_cap_device, no_cap_op = ac_device(
    fixed_reverse_supply, power=power_levels[0], tau_rec=transport_tau_rec,
    base_circuit=no_cap_circuit, **transport_settings)
no_cap_response = optical_ac_response(no_cap_device, no_cap_op, ac_freqs, power=power_levels[0])["carrier"]
with_cap = parasitic_ac_results[power_levels[0]]
np.testing.assert_allclose(no_cap_device.port(no_cap_op, "carrier"),
                           with_cap["device"].port(with_cap["op"], "carrier"), rtol=1e-7, atol=1e-12)
np.testing.assert_allclose(no_cap_response[0], with_cap["response"]["carrier"][0], rtol=1e-7)
assert bandwidth_3db(ac_freqs, no_cap_response / no_cap_response[0]) > with_cap["bandwidth"]
print("Subcircuit topology, series voltage drop, pad leakage division, capacitor DC invariance, and parasitic bandwidth checks passed.")


# The same package interface also supports a true device-only topology:
# with_parasitics=False removes R_series, R_pad, and C_pad entirely.
bare_package_circuit = make_parasitic_fixture(with_parasitics=False)
assert "PD_carrier~R_series,p1" not in bare_package_circuit.port_map
bare_powers = power_levels[[0, -1]]
bare_sweep = optical_ac_sweep(
    bare_package_circuit, fixed_reverse_supply, jnp.asarray(bare_powers),
    tau_rec=transport_tau_rec, **transport_settings,
)
for k, power in enumerate(bare_powers):
    bare_response = jax.tree.map(lambda values, k=k: values[k], bare_sweep["response"])
    for name in ("ideal", "diode", "carrier"):
        np.testing.assert_allclose(bare_response[name], power_ac_results[0.0][power]["response"][name],
                                   rtol=1e-6, atol=1e-9)
print("Packaged device-only response matches the intrinsic component circuit.")


## Periodic optical modulation with harmonic balance

For a periodic experiment, replace the Gaussian sources with `OpticalCW`, which supports sinusoidal **incident power**. Keep the same detector-only netlist, matched optical terminations, DC bias and 50 Ω loads. `hb()` solves the complete nonlinear periodic steady state, including the carrier populations and junction charge.

First compare a small modulation with the SP-derived differential gain at 10 GHz. Then increase modulation depth to 50% and inspect the harmonics generated by transport/screening. A cosine of power amplitude $mP$ has positive-frequency Fourier coefficient $mP/2$, so divide the solved load-current fundamental by that coefficient to get A/W. Harmonic amplitudes use twice the magnitude of the normalized Fourier coefficients. Repeat the large-signal solve with more harmonics to check convergence.


In [ ]:
hb_frequency = 10e9
hb_power = power_levels[-1]
hb_circuit = compile_circuit(
    make_testbench(optical_source="optical_cw"), {**models, "optical_cw": OpticalCW},
    is_complex=True, backend="dense", rtol=1e-9, atol=1e-12,
)
hb_params = {"BIAS.V": fixed_reverse_supply, "power": hb_power, "frequency": hb_frequency,
             "screening": transport_settings["screening"], "tau_rec": transport_tau_rec,
             "tau_e0": transport_settings["tau_e0"], "tau_h0": transport_settings["tau_h0"]}
hb_op = hb_circuit.dc(params={**hb_params, "modulation": 0.0}, rtol=1e-10, atol=1e-13)
small_modulation = 1e-3
small_time, small_spectrum = hb_circuit.hb(
    freq=hb_frequency, harmonics=5, y0=hb_op,
    params={**hb_params, "modulation": small_modulation}, max_steps=100,
)
# Use the same CW operating point and parameters in the public SP solver.
sp_gain = optical_ac_response(
    hb_circuit, hb_op, jnp.array([hb_frequency]), power=hb_power,
    params={**hb_params, "modulation": 0.0},
)
for name in detector_names:
    fundamental_current = (hb_circuit.port(small_spectrum, name)
                           - hb_circuit.port(small_spectrum, "GND,p1"))[1] / 50
    hb_gain = fundamental_current / (hb_power * small_modulation / 2)
    np.testing.assert_allclose(hb_gain, sp_gain[name][0], rtol=0.002, atol=1e-6)

large_modulation = 0.5
large_time, large_spectrum = hb_circuit.hb(
    freq=hb_frequency, harmonics=9, y0=hb_op,
    params={**hb_params, "modulation": large_modulation}, max_steps=100,
)
refined_time, refined_spectrum = hb_circuit.hb(
    freq=hb_frequency, harmonics=17, y0=hb_op,
    params={**hb_params, "modulation": large_modulation}, max_steps=100,
)
large_current = (hb_circuit.port(large_spectrum, "carrier")
                 - hb_circuit.port(large_spectrum, "GND,p1")) / 50
refined_current = (hb_circuit.port(refined_spectrum, "carrier")
                   - hb_circuit.port(refined_spectrum, "GND,p1")) / 50
np.testing.assert_allclose(large_current[:4], refined_current[:4], rtol=0.01, atol=1e-9)
assert np.all(np.isfinite(refined_time))
assert abs(refined_current[2]) > 1e-3 * abs(refined_current[1])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
period_ps = 1e12 / hb_frequency
sample_times_ps = np.arange(len(refined_time)) * period_ps / len(refined_time)
for name in detector_names:
    load_current = np.real(hb_circuit.port(refined_time, name)
                            - hb_circuit.port(refined_time, "GND,p1")) / 50
    axes[0].plot(sample_times_ps, load_current * 1e3, "o-", ms=3, label=name)
axes[0].set(xlabel="Time within one period (ps)", ylabel="Load current (mA)",
            title="50% incident-power modulation")
axes[0].legend()
orders = np.arange(1, 10)
axes[1].stem(orders, 2 * np.abs(refined_current[1:10]) * 1e6)
axes[1].set(xlabel="Harmonic order", ylabel="Carrier-model current amplitude (µA)", yscale="log")
plt.tight_layout()
plt.show()
print("HB small-signal/SP agreement and large-signal harmonic convergence checks passed.")


## Next steps

Fitting order, if you're calibrating this to a real device: complex optical reflection and external responsivity vs. wavelength first (separating absorption loss from conversion efficiency), then dark I-V and C-V, then low-power carrier sweep-out times, then screening/recombination from power and bias sweeps. Things this notebook doesn't cover: measured profiles, series resistance beyond the simple model above, trapping, avalanche multiplication, spatial drift-diffusion. The two-population approximation here can't reproduce a real spatial transit-time distribution — some of the short-pulse rise you see is really just the stimulus and load RC, not the device.

Once the profiles and transport approximation are agreed on, promote the decorated components, shared helpers, and the subcircuit factory into `circulax.components.photonic`, with this notebook kept as the worked example. Keep the intrinsic junction model separate from the packaging so contacts/metal/pads/bias/load can all be swapped independently.
